[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/16-efficient-and-long-attention.ipynb)

# Module 16: Efficient and Long Attention

Companion notebook to [Module 16](https://transformers-and-math.drmhse.com/modules/16-efficient-and-long-attention) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — FlashAttention's online softmax (section 16.2)
Process keys in tiles, keeping a running maximum, sum and output per query. The full score matrix is never stored.

In [ ]:
def tiled_attention(Q, K, V, tile=16):
    N, d = Q.shape
    m = torch.full((N, 1), float('-inf'))
    l = torch.zeros(N, 1)
    o = torch.zeros(N, V.shape[1])
    for start in range(0, K.shape[0], tile):
        s = Q @ K[start:start + tile].T / math.sqrt(d)          # one tile of scores
        m_new = torch.maximum(m, s.max(1, keepdim=True).values)
        p = torch.exp(s - m_new)
        scale = torch.exp(m - m_new)                             # rescale what we had to the new maximum
        l = l * scale + p.sum(1, keepdim=True)
        o = o * scale + p @ V[start:start + tile]
        m = m_new
    return o / l

Q, K, V = torch.randn(128, 64), torch.randn(512, 64), torch.randn(512, 64)
exact = torch.softmax(Q @ K.T / 8, -1) @ V
for tile in (4, 16, 128):
    print(f'tile {tile:3}: max difference from exact attention {(tiled_attention(Q, K, V, tile) - exact).abs().max():.2e}')

# Without the running maximum, large scores overflow.
big = 40 * Q
naive = torch.exp(big @ K.T / 8) @ V / torch.exp(big @ K.T / 8).sum(1, keepdim=True)
print('naive with large scores has nan/inf:', bool(torch.isnan(naive).any() or torch.isinf(naive).any()),
      '| tiled is still exact:', torch.allclose(tiled_attention(big, K, V), torch.softmax(big @ K.T / 8, -1) @ V, atol=1e-5))

## Part 2 — Local (sliding-window) attention (section 16.3)

In [ ]:
def sliding_mask(N, W):
    i, j = torch.arange(N)[:, None], torch.arange(N)[None, :]
    allowed = (j <= i) & (j > i - W)
    return torch.where(allowed, 0.0, float('-inf'))

print(sliding_mask(8, 3))
ctx, W = 131072, 1024
share = (5 * W + ctx) / (6 * ctx)
print(f'Gemma 3 pattern (5 local : 1 global, W={W}) at {ctx:,} tokens keeps {share:.1%} of an all-global cache')

## Part 3 — Logit soft-capping (section 16.5)

In [ ]:
z = torch.tensor([5., 30., 100., 300.], dtype=torch.float64, requires_grad=True)
capped = 30 * torch.tanh(z / 30)
capped.sum().backward()
for zi, ci, gi in zip(z.tolist(), capped.tolist(), z.grad.tolist()):
    print(f'z={zi:6.1f}  →  {ci:6.2f}   slope {gi:.2e}')

## Your turn
1. Count the bytes of scores the exact computation materializes for N = 32,768 queries and keys in fp16, and compare with the tiled version's per-row state.
2. With a window of 4,096 and every layer local, how far back can the last token see after 8 layers?